# 0.4秒 BC 100轮续训 · Warmup/Cosine · W&B

在 Colab 选择 **GPU（推荐 L4）**，按顺序运行下面 **6个代码单元格**。
保持现有183M架构和训练目标，自动恢复同一数据阶段的最新完整checkpoint。

- W&B：在线记录 train / validation loss、选牌、动作召回、梯度、FP16 scale、内存。
- Notebook：显示轨迹座位进度、epoch、全局step、loss和ETA。父阶段256步不冒充本阶段进度。
- 数据：只缓存当前manifest引用的压缩shards、元数据、源码和一个最新checkpoint；训练读`/content`。
- 学习率：350次成功更新预热至峰值3e-5，随后余弦衰减到3e-6；恢复保留调度位置。
- 保存：每128个座位及epoch结束原子保存并校验备份latest；逐局解压缓存自动回收。

**中断了显示：只重跑代码5。** Trainer继续在后台；重复运行代码5会连接已有进程。
新运行时从代码1开始，自动挑选原训练目录、本notebook备份和本地缓存中最新的有效恢复点。
`TARGET_EPOCHS=100`表示这个数据阶段的总目标，含已完成的epoch。余弦调度开始后保持相同100轮终点。


## 代码1 / 6：挂载Drive和设置目标

默认路径指向当前0.4秒BC阶段。本notebook的续训结果写入其`bc100_cosine`子目录。
首次运行Google会要求挂载授权；W&B登录在代码3完成，密钥只输入密码框或Colab Secret。


In [ ]:
from pathlib import Path
import os, sys, json, time
from google.colab import drive

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

SOURCE_RUN = Path('/content/drive/MyDrive/cr_rl/runs/state_global_event040_20261110')
NOTEBOOK_RUN = SOURCE_RUN / 'bc100_cosine'
WORK = Path('/content/crrl_bc100_notebook')
TARGET_EPOCHS = 100  # 当前数据阶段总目标，包含已完成epoch
WARMUP_STEPS = 350
MIN_LEARNING_RATE = 3e-6
FRAME_MICROBATCH = 32
FUSED_OPTIMIZER = True
FLOAT32_MATMUL_PRECISION = 'high'
DEFER_METRICS = True
CHECKPOINT_EVERY_SEATS = 128
VALIDATION_EVERY_SEATS = 350
WANDB_PROJECT = 'cr-rl'
WANDB_ENTITY = None  # 留空使用登录账号默认空间；团队账号可填team slug
WANDB_NAME = 'bc040-183m-100ep-cosine'

if not (SOURCE_RUN / 'dataset/dataset.manifest.json').is_file():
    raise FileNotFoundError('未找到当前阶段的数据manifest。请确认挂载的是训练使用的Google账号，或修改SOURCE_RUN。')
WORK.mkdir(parents=True, exist_ok=True)
NOTEBOOK_RUN.mkdir(parents=True, exist_ok=True)
print('数据来源:', SOURCE_RUN)
print('续训备份:', NOTEBOOK_RUN)
print('目标总epoch:', TARGET_EPOCHS)


## 代码2 / 6：检查GPU，加载notebook工具

只补缺少的W&B、tqdm、PyYAML，沿用Colab现有Torch。CUDA不可用会在加载大文件前停止。


In [ ]:
import importlib.util, subprocess, base64, hashlib
import torch

if not torch.version.cuda or not torch.cuda.is_available():
    raise RuntimeError('当前不是可用的GPU运行时。请在Colab“更改运行时类型”选择GPU，再从代码1运行；无需重装Torch。')
missing = [package for module, package in [
    ('wandb', 'wandb>=0.19,<1'), ('tqdm', 'tqdm>=4.66,<5'), ('yaml', 'PyYAML>=6,<7')
] if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--disable-pip-version-check', *missing], check=True)
TOOLS = WORK / 'notebook_tools'
TOOLS.mkdir(exist_ok=True)
EMBEDDED_MODULES = {"notebook_state_bc.py":{"base64":"IiIiTm90ZWJvb2sgcHJvZ3Jlc3MgYW5kIFcmQiBtaXJyb3JpbmcgZm9yIHRoZSBleGlzdGluZyBzdGF0ZS1CQyBKU09OTCBsb2cuCgpUaGlzIG1vZHVsZSBuZXZlciBpbXBvcnRzIFRvcmNoLCBzdGFydHMgYSB0cmFpbmVyLCBvciBjcmVhdGVzIGEgVyZCIHJ1bi4gVGhlCm5vdGVib29rIG93bnMgdGhvc2Ugb3BlcmF0aW9ucy4gTG9nIGRlbGl2ZXJ5IGlzIGF0IGxlYXN0IG9uY2U6IHRoZSBkdXJhYmxlCmN1cnNvciBhZHZhbmNlcyBvbmx5IGFmdGVyIGBgcnVuLmxvZ2BgIGFjY2VwdHMgdGhlIGV2ZW50LiBBIGtlcm5lbCBjcmFzaCBpbiB0aGUKc21hbGwgaW50ZXJ2YWwgYmV0d2VlbiBkZWxpdmVyeSBhbmQgY3Vyc29yIHNhdmUgY2FuIHJlcGxheSB0aGF0IGZpbmFsIGV2ZW50LgoiIiIKZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwoKaW1wb3J0IGhhc2hsaWIKaW1wb3J0IGpzb24KaW1wb3J0IG1hdGgKaW1wb3J0IG9zCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAppbXBvcnQgdGltZQppbXBvcnQgd2FybmluZ3MKCgpkZWYgX2ludGVnZXIodmFsdWUsIG5hbWUsIG1pbmltdW09MCk6CiAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBib29sKSBvciBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgaW50KSBvciB2YWx1ZSA8IG1pbmltdW06CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIntuYW1lfSBtdXN0IGJlIGFuIGludGVnZXIgPj0ge21pbmltdW19IikKICAgIHJldHVybiB2YWx1ZQoKCmRlZiByZXN1bWVfY29tcGxldGVkKHByb2dyZXNzLCBzZWF0c19wZXJfZXBvY2gpOgogICAgIiIiQ2hlY2twb2ludCBlcG9jaHMgYXJlIHplcm8gYmFzZWQ7IG5leHRfZXBpc29kZSBjb3VudHMgY29tcGxldGVkIHNlYXRzLiIiIgogICAgc2VhdHMgPSBfaW50ZWdlcihzZWF0c19wZXJfZXBvY2gsICJzZWF0c19wZXJfZXBvY2giLCAxKQogICAgZXBvY2ggPSBfaW50ZWdlcihwcm9ncmVzcy5nZXQoImVwb2NoIiksICJlcG9jaCIpCiAgICBuZXh0X2VwaXNvZGUgPSBfaW50ZWdlcihwcm9ncmVzcy5nZXQoIm5leHRfZXBpc29kZSIpLCAibmV4dF9lcGlzb2RlIikKICAgIGlmIG5leHRfZXBpc29kZSA+IHNlYXRzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNoZWNrcG9pbnQgbmV4dF9lcGlzb2RlIGV4Y2VlZHMgdGhpcyBkYXRhc2V0J3MgZXBvY2giKQogICAgcmV0dXJuIGVwb2NoICogc2VhdHMgKyBuZXh0X2VwaXNvZGUKCgpkZWYgY29tcGxldGVkX3NlYXRzKHJlY29yZCwgc2VhdHNfcGVyX2Vwb2NoKToKICAgICIiIlRyYWluIHJvd3MgdXNlIG9uZS1iYXNlZCBlcG9jaCBhbmQgYSB6ZXJvLWJhc2VkIGNvbXBsZXRlZCBlcGlzb2RlIGluZGV4LiIiIgogICAgaWYgcmVjb3JkLmdldCgicGhhc2UiKSAhPSAic3RhdGVfYmNfdHJhaW4iOgogICAgICAgIHJldHVybiBOb25lICAjIEluaXRpYWwvbWlkLWVwb2NoIHZhbGlkYXRpb24gaXMgbm90IGFuIGVwb2NoIGNvbXBsZXRpb24uCiAgICBzZWF0cyA9IF9pbnRlZ2VyKHNlYXRzX3Blcl9lcG9jaCwgInNlYXRzX3Blcl9lcG9jaCIsIDEpCiAgICBlcG9jaCA9IF9pbnRlZ2VyKHJlY29yZC5nZXQoImVwb2NoIiksICJlcG9jaCIsIDEpCiAgICBpbmRleCA9IF9pbnRlZ2VyKHJlY29yZC5nZXQoImVwaXNvZGVfaW5kZXgiKSwgImVwaXNvZGVfaW5kZXgiKQogICAgaWYgaW5kZXggPj0gc2VhdHM6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTG9nZ2VkIGVwaXNvZGUgaW5kZXggZXhjZWVkcyB0aGlzIGRhdGFzZXQncyBlcG9jaCIpCiAgICByZXR1cm4gKGVwb2NoIC0gMSkgKiBzZWF0cyArIGluZGV4ICsgMQoKCmRlZiBtZXRyaWNfcGF5bG9hZChyZWNvcmQpOgogICAgIiIiTWFwIGZpbml0ZSBzY2FsYXIgbWV0cmljczsgZG8gbm90IGNvbXBhcmUgVyZCJ3Mgb3duIHN0ZXAgdG8gbW9kZWwgc3RlcHMuIiIiCiAgICBwaGFzZSA9IHJlY29yZC5nZXQoInBoYXNlIikKICAgIHByZWZpeCA9IHsic3RhdGVfYmNfdHJhaW4iOiAidHJhaW4iLCAic3RhdGVfYmNfdmFsaWRhdGlvbiI6ICJ2YWxpZGF0aW9uIn0uZ2V0KHBoYXNlKQogICAgaWYgcHJlZml4IGlzIE5vbmU6CiAgICAgICAgaWYgcGhhc2UgIT0gInN0YXRlX2JjX3N0YXJ0IjoKICAgICAgICAgICAgcmV0dXJuIHt9CiAgICAgICAgcmVzdWx0ID0ge30KICAgICAgICBmb3Iga2V5IGluICgicGFyYW1ldGVyX2NvdW50IiwgInRyYWluYWJsZV9wYXJhbWV0ZXJfY291bnQiLCAicmFtX2xpbWl0X2dpYiIpOgogICAgICAgICAgICB2YWx1ZSA9IHJlY29yZC5nZXQoa2V5KQogICAgICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCAoaW50LCBmbG9hdCkpIGFuZCBub3QgaXNpbnN0YW5jZSh2YWx1ZSwgYm9vbCkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICAgICAgcmVzdWx0W2YicnVuL3trZXl9Il0gPSB2YWx1ZQogICAgICAgIHJldHVybiByZXN1bHQKICAgIHN0ZXAgPSBfaW50ZWdlcihyZWNvcmQuZ2V0KCJvcHRpbWl6ZXJfc3RlcHMiKSwgIm9wdGltaXplcl9zdGVwcyIpCiAgICByZXN1bHQgPSB7InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIjogc3RlcH0KICAgIGZvciBrZXksIHZhbHVlIGluIHJlY29yZC5pdGVtcygpOgogICAgICAgIGlmIGtleSBpbiB7Im9wdGltaXplcl9zdGVwcyIsICJlbGFwc2VkX3NlY29uZHMifToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBpc2luc3RhbmNlKHZhbHVlLCBib29sKToKICAgICAgICAgICAgcmVzdWx0W2Yie3ByZWZpeH0ve2tleX0iXSA9IGludCh2YWx1ZSkKICAgICAgICBlbGlmIGlzaW5zdGFuY2UodmFsdWUsIChpbnQsIGZsb2F0KSkgYW5kIG1hdGguaXNmaW5pdGUodmFsdWUpOgogICAgICAgICAgICByZXN1bHRbZiJ7cHJlZml4fS97a2V5fSJdID0gdmFsdWUKICAgIHJldHVybiByZXN1bHQKCgpkZWYgY29uZmlndXJlX3dhbmRiKHJ1bik6CiAgICAiIiJBIGN1c3RvbSB4IGF4aXMgcHJlc2VydmVzIHRyYWluIGFuZCB2YWxpZGF0aW9uIGV2ZW50cyBhdCB0aGUgc2FtZSBzdGVwLiIiIgogICAgcnVuLmRlZmluZV9tZXRyaWMoInRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIikKICAgIHJ1bi5kZWZpbmVfbWV0cmljKCJ0cmFpbi8qIiwgc3RlcF9tZXRyaWM9InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIiwgc3RlcF9zeW5jPVRydWUpCiAgICBydW4uZGVmaW5lX21ldHJpYygidmFsaWRhdGlvbi8qIiwgc3RlcF9tZXRyaWM9InRyYWluZXIvb3B0aW1pemVyX3N0ZXBzIiwgc3RlcF9zeW5jPVRydWUpCgoKY2xhc3MgSlNPTkxUYWlsZXI6CiAgICAiIiJSZWFkIGNvbXBsZXRlIGFwcGVuZGVkIGxpbmVzIGFuZCBwZXJzaXN0IGV4cGxpY2l0bHkgYWNrbm93bGVkZ2VkIG9mZnNldHMuCgogICAgQ3Vyc29yIGlkZW50aXR5IHNob3VsZCBiZSB0aGUgVyZCIHJ1biBJRC4gUHJlZml4IGFuZCBuZWFyLW9mZnNldCBhbmNob3JzCiAgICBhbGxvdyBpZGVudGljYWwgRHJpdmUtcmVzdG9yZWQgY29waWVzIHdoaWxlIGRldGVjdGluZyB0cnVuY2F0aW9uIG9yIGxvZwogICAgcmVwbGFjZW1lbnQuIElmIHRoZSBsb2cgcm9sbHMgYmFjaywgcmVwbGF5IGlzIGV4cGxpY2l0IHJhdGhlciB0aGFuIHNpbGVudGx5CiAgICBza2lwcGluZyBvbGQgYnl0ZXMuIEN1cnNvciBmaWxlcyBhcmUgc21hbGwgSlNPTiwgbmV2ZXIgbW9kZWwvZGF0YSB0ZW5zb3JzLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHBhdGgsIGN1cnNvcl9wYXRoPU5vbmUsICosIGlkZW50aXR5PSJkZWZhdWx0Iik6CiAgICAgICAgc2VsZi5wYXRoID0gUGF0aChwYXRoKQogICAgICAgIHNlbGYuY3Vyc29yX3BhdGggPSBQYXRoKGN1cnNvcl9wYXRoKSBpZiBjdXJzb3JfcGF0aCBpcyBub3QgTm9uZSBlbHNlIE5vbmUKICAgICAgICBzZWxmLmlkZW50aXR5ID0gc3RyKGlkZW50aXR5KQogICAgICAgIHNlbGYub2Zmc2V0ID0gMAogICAgICAgIHNlbGYucHJlZml4X2xlbmd0aCA9IDAKICAgICAgICBzZWxmLnByZWZpeF9zaGEyNTYgPSBOb25lCiAgICAgICAgc2VsZi5hbmNob3Jfc3RhcnQgPSAwCiAgICAgICAgc2VsZi5hbmNob3Jfc2hhMjU2ID0gTm9uZQogICAgICAgIHNlbGYucmVzZXRzID0gMAogICAgICAgIGlmIHNlbGYuY3Vyc29yX3BhdGggaXMgbm90IE5vbmUgYW5kIHNlbGYuY3Vyc29yX3BhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHNhdmVkID0ganNvbi5sb2FkcyhzZWxmLmN1cnNvcl9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKICAgICAgICAgICAgaWYgc2F2ZWQuZ2V0KCJpZGVudGl0eSIpID09IHNlbGYuaWRlbnRpdHk6CiAgICAgICAgICAgICAgICBzZWxmLm9mZnNldCA9IF9pbnRlZ2VyKHNhdmVkLmdldCgib2Zmc2V0IiksICJjdXJzb3Igb2Zmc2V0IikKICAgICAgICAgICAgICAgIHNlbGYucHJlZml4X2xlbmd0aCA9IF9pbnRlZ2VyKHNhdmVkLmdldCgicHJlZml4X2xlbmd0aCIsIDApLCAicHJlZml4X2xlbmd0aCIpCiAgICAgICAgICAgICAgICBzZWxmLnByZWZpeF9zaGEyNTYgPSBzYXZlZC5nZXQoInByZWZpeF9zaGEyNTYiKQogICAgICAgICAgICAgICAgc2VsZi5hbmNob3Jfc3RhcnQgPSBfaW50ZWdlcihzYXZlZC5nZXQoImFuY2hvcl9zdGFydCIsIDApLCAiYW5jaG9yX3N0YXJ0IikKICAgICAgICAgICAgICAgIHNlbGYuYW5jaG9yX3NoYTI1NiA9IHNhdmVkLmdldCgiYW5jaG9yX3NoYTI1NiIpCgogICAgQHN0YXRpY21ldGhvZAogICAgZGVmIF9kaWdlc3Qoc3RyZWFtLCBzdGFydCwgbGVuZ3RoKToKICAgICAgICBzdHJlYW0uc2VlayhzdGFydCkKICAgICAgICB2YWx1ZSA9IHN0cmVhbS5yZWFkKGxlbmd0aCkKICAgICAgICByZXR1cm4gaGFzaGxpYi5zaGEyNTYodmFsdWUpLmhleGRpZ2VzdCgpCgogICAgZGVmIF9yZXNldChzZWxmKToKICAgICAgICBzZWxmLm9mZnNldCA9IDAKICAgICAgICBzZWxmLnByZWZpeF9sZW5ndGggPSAwCiAgICAgICAgc2VsZi5wcmVmaXhfc2hhMjU2ID0gTm9uZQogICAgICAgIHNlbGYuYW5jaG9yX3N0YXJ0ID0gMAogICAgICAgIHNlbGYuYW5jaG9yX3NoYTI1NiA9IE5vbmUKICAgICAgICBzZWxmLnJlc2V0cyArPSAxCiAgICAgICAgd2FybmluZ3Mud2FybigiVHJhaW5pbmcgbG9nIHdhcyB0cnVuY2F0ZWQvcmVwbGFjZWQ7IHJlcGxheWluZyBpdHMgYXZhaWxhYmxlIGV2ZW50cyIsIFJ1bnRpbWVXYXJuaW5nKQoKICAgIGRlZiBwZW5kaW5nKHNlbGYpOgogICAgICAgICIiIllpZWxkIChyZWNvcmQsIGJ5dGVfb2Zmc2V0KTsgYWNrbm93bGVkZ2UgZWFjaCBvbmx5IGFmdGVyIGRlbGl2ZXJ5LiIiIgogICAgICAgIGlmIG5vdCBzZWxmLnBhdGguZXhpc3RzKCk6CiAgICAgICAgICAgIHJldHVybgogICAgICAgIHdpdGggc2VsZi5wYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgICAgICBzaXplID0gb3MuZnN0YXQoc3RyZWFtLmZpbGVubygpKS5zdF9zaXplCiAgICAgICAgICAgIGNoYW5nZWQgPSBzaXplIDwgc2VsZi5vZmZzZXQKICAgICAgICAgICAgaWYgc2VsZi5wcmVmaXhfbGVuZ3RoOgogICAgICAgICAgICAgICAgY2hhbmdlZCB8PSBzaXplIDwgc2VsZi5wcmVmaXhfbGVuZ3RoIG9yIHNlbGYuX2RpZ2VzdChzdHJlYW0sIDAsIHNlbGYucHJlZml4X2xlbmd0aCkgIT0gc2VsZi5wcmVmaXhfc2hhMjU2CiAgICAgICAgICAgIGlmIHNlbGYub2Zmc2V0IGFuZCBzZWxmLmFuY2hvcl9zaGEyNTY6CiAgICAgICAgICAgICAgICBjaGFuZ2VkIHw9IHNlbGYuX2RpZ2VzdChzdHJlYW0sIHNlbGYuYW5jaG9yX3N0YXJ0LCBzZWxmLm9mZnNldCAtIHNlbGYuYW5jaG9yX3N0YXJ0KSAhPSBzZWxmLmFuY2hvcl9zaGEyNTYKICAgICAgICAgICAgaWYgY2hhbmdlZDoKICAgICAgICAgICAgICAgIHNlbGYuX3Jlc2V0KCkKICAgICAgICAgICAgaWYgbm90IHNlbGYucHJlZml4X2xlbmd0aCBhbmQgc2l6ZToKICAgICAgICAgICAgICAgIHNlbGYucHJlZml4X2xlbmd0aCA9IG1pbihzaXplLCAxMDI0KQogICAgICAgICAgICAgICAgc2VsZi5wcmVmaXhfc2hhMjU2ID0gc2VsZi5fZGlnZXN0KHN0cmVhbSwgMCwgc2VsZi5wcmVmaXhfbGVuZ3RoKQogICAgICAgICAgICBzdHJlYW0uc2VlayhzZWxmLm9mZnNldCkKICAgICAgICAgICAgd2hpbGUgVHJ1ZToKICAgICAgICAgICAgICAgIHN0YXJ0ID0gc3RyZWFtLnRlbGwoKQogICAgICAgICAgICAgICAgbGluZSA9IHN0cmVhbS5yZWFkbGluZSg4ICogMTAyNCoqMiArIDEpCiAgICAgICAgICAgICAgICBpZiBsZW4obGluZSkgPiA4ICogMTAyNCoqMjoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiVHJhaW5pbmcgSlNPTkwgbGluZSBleGNlZWRzIDhNaUIgYXQgYnl0ZSB7c3RhcnR9IikKICAgICAgICAgICAgICAgIGlmIG5vdCBsaW5lIG9yIG5vdCBsaW5lLmVuZHN3aXRoKGIiXG4iKToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gICMgQSBjb25jdXJyZW50IHdyaXRlciBjYW4gZmluaXNoIHRoaXMgcm93IGxhdGVyLgogICAgICAgICAgICAgICAgZW5kID0gc3RyZWFtLnRlbGwoKQogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJlY29yZCA9IGpzb24ubG9hZHMobGluZSkKICAgICAgICAgICAgICAgIGV4Y2VwdCAoVmFsdWVFcnJvciwgVW5pY29kZURlY29kZUVycm9yKSBhcyBlcnJvcjoKICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiSW52YWxpZCBjb21wbGV0ZSB0cmFpbmluZyBKU09OTCBsaW5lIGF0IGJ5dGUge3N0YXJ0fSIpIGZyb20gZXJyb3IKICAgICAgICAgICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKHJlY29yZCwgZGljdCk6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmIlRyYWluaW5nIEpTT05MIHJlY29yZCBhdCBieXRlIHtzdGFydH0gaXMgbm90IGFuIG9iamVjdCIpCiAgICAgICAgICAgICAgICB5aWVsZCByZWNvcmQsIGVuZAoKICAgIGRlZiBhY2tub3dsZWRnZShzZWxmLCBvZmZzZXQpOgogICAgICAgIG9mZnNldCA9IF9pbnRlZ2VyKG9mZnNldCwgImFja25vd2xlZGdlZCBvZmZzZXQiKQogICAgICAgIGlmIG9mZnNldCA8IHNlbGYub2Zmc2V0OgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJDYW5ub3QgYWNrbm93bGVkZ2UgYSBwcmV2aW91cyBsb2cgcG9zaXRpb24iKQogICAgICAgIHdpdGggc2VsZi5wYXRoLm9wZW4oInJiIikgYXMgc3RyZWFtOgogICAgICAgICAgICBpZiBvcy5mc3RhdChzdHJlYW0uZmlsZW5vKCkpLnN0X3NpemUgPCBvZmZzZXQ6CiAgICAgICAgICAgICAgICByYWlzZSBPU0Vycm9yKCJUcmFpbmluZyBsb2cgdHJ1bmNhdGVkIGJlZm9yZSBhY2tub3dsZWRnZW1lbnQiKQogICAgICAgICAgICBzZWxmLmFuY2hvcl9zdGFydCA9IG1heCgwLCBvZmZzZXQgLSAxMjgpCiAgICAgICAgICAgIHNlbGYuYW5jaG9yX3NoYTI1NiA9IHNlbGYuX2RpZ2VzdChzdHJlYW0sIHNlbGYuYW5jaG9yX3N0YXJ0LCBvZmZzZXQgLSBzZWxmLmFuY2hvcl9zdGFydCkKICAgICAgICBzZWxmLm9mZnNldCA9IG9mZnNldAogICAgICAgIGlmIHNlbGYuY3Vyc29yX3BhdGggaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuY3Vyc29yX3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgdGVtcG9yYXJ5ID0gc2VsZi5jdXJzb3JfcGF0aC53aXRoX25hbWUoc2VsZi5jdXJzb3JfcGF0aC5uYW1lICsgIi50bXAiKQogICAgICAgICAgICB0ZW1wb3Jhcnkud3JpdGVfdGV4dChqc29uLmR1bXBzKHsKICAgICAgICAgICAgICAgICJpZGVudGl0eSI6IHNlbGYuaWRlbnRpdHksICJvZmZzZXQiOiBzZWxmLm9mZnNldCwKICAgICAgICAgICAgICAgICJwcmVmaXhfbGVuZ3RoIjogc2VsZi5wcmVmaXhfbGVuZ3RoLCAicHJlZml4X3NoYTI1NiI6IHNlbGYucHJlZml4X3NoYTI1NiwKICAgICAgICAgICAgICAgICJhbmNob3Jfc3RhcnQiOiBzZWxmLmFuY2hvcl9zdGFydCwgImFuY2hvcl9zaGEyNTYiOiBzZWxmLmFuY2hvcl9zaGEyNTYsCiAgICAgICAgICAgIH0pLCBlbmNvZGluZz0idXRmLTgiKQogICAgICAgICAgICB0ZW1wb3JhcnkucmVwbGFjZShzZWxmLmN1cnNvcl9wYXRoKQoKCmRlZiBsb2dfdGFpbChwYXRoLCBsaW5lcz02MCwgbWF4X2J5dGVzPTY1NTM2KToKICAgIHBhdGggPSBQYXRoKHBhdGgpCiAgICBpZiBub3QgcGF0aC5leGlzdHMoKToKICAgICAgICByZXR1cm4gIih0cmFpbmVyIGxvZyBkb2VzIG5vdCBleGlzdCkiCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgc3RyZWFtLnNlZWsobWF4KDAsIHBhdGguc3RhdCgpLnN0X3NpemUgLSBtYXhfYnl0ZXMpKQogICAgICAgIHJldHVybiAiXG4iLmpvaW4oc3RyZWFtLnJlYWQoKS5kZWNvZGUoInV0Zi04IiwgZXJyb3JzPSJyZXBsYWNlIikuc3BsaXRsaW5lcygpWy1saW5lczpdKQoKCmRlZiBtb25pdG9yX3Byb2Nlc3MocHJvY2VzcywgbWV0cmljc19wYXRoLCB0cmFpbl9sb2dfcGF0aCwgKiwgcnVuPU5vbmUsIHRhcmdldF9lcG9jaHM9MywKICAgICAgICAgICAgICAgICAgICBzZWF0c19wZXJfZXBvY2g9MzUwLCBpbml0aWFsX2NvbXBsZXRlZD0wLCBjdXJzb3JfcGF0aD1Ob25lLAogICAgICAgICAgICAgICAgICAgIHBvbGxfc2Vjb25kcz0yLCBiYXI9Tm9uZSwgb25fcG9sbD1Ob25lKToKICAgICIiIk1pcnJvciBsb2dzIGFuZCBtb25pdG9yIGFuIGV4aXN0aW5nIHN1YnByb2Nlc3Mgd2l0aG91dCBhbHRlcmluZyB0cmFpbmluZy4KCiAgICBLZXlib2FyZEludGVycnVwdCBjbG9zZXMgdGhlIGRpc3BsYXkgYW5kIHByb3BhZ2F0ZXMuIFRoZSBub3RlYm9vayBtdXN0CiAgICBleHBsaWNpdGx5IGRlY2lkZSB3aGV0aGVyIGl0cyBvd25lZCB0cmFpbmVyIHNob3VsZCBzdG9wIG9yIGNvbnRpbnVlLgogICAgVGVtcG9yYXJ5IFcmQiBkZWxpdmVyeSBmYWlsdXJlcyBsZWF2ZSB0aGVpciBjdXJzb3IgcGVuZGluZyBhbmQgYXJlIHJldHJpZWQKICAgIHdoaWxlIHRyYWluaW5nIGNvbnRpbnVlcy4gT24gc3VjY2Vzc2Z1bCB0cmFpbmVyIGV4aXQsIHVucmVzb2x2ZWQgZGVsaXZlcnkKICAgIHJldHVybnMgYGBsb2dnaW5nX3N0YXR1cz0ncGVuZGluZydgYCBhbmQgcHJpbnRzIGFuIGV4cGxpY2l0IHdhcm5pbmcuCiAgICBgYHJ1bi5maW5pc2hgYCBpcyBhbHNvIHRoZSBub3RlYm9vaydzIHJlc3BvbnNpYmlsaXR5LgogICAgIiIiCiAgICB0b3RhbCA9IF9pbnRlZ2VyKHRhcmdldF9lcG9jaHMsICJ0YXJnZXRfZXBvY2hzIiwgMSkgKiBfaW50ZWdlcihzZWF0c19wZXJfZXBvY2gsICJzZWF0c19wZXJfZXBvY2giLCAxKQogICAgaW5pdGlhbCA9IF9pbnRlZ2VyKGluaXRpYWxfY29tcGxldGVkLCAiaW5pdGlhbF9jb21wbGV0ZWQiKQogICAgaWYgaW5pdGlhbCA+IHRvdGFsOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoIkNoZWNrcG9pbnQgcHJvZ3Jlc3MgZXhjZWVkcyByZXF1ZXN0ZWQgdGFyZ2V0IGVwb2NocyIpCiAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShwb2xsX3NlY29uZHMpIG9yIHBvbGxfc2Vjb25kcyA8PSAwOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoInBvbGxfc2Vjb25kcyBtdXN0IGJlIHBvc2l0aXZlIikKICAgIGlmIGJhciBpcyBOb25lOgogICAgICAgIGZyb20gdHFkbS5hdXRvIGltcG9ydCB0cWRtCiAgICAgICAgYmFyID0gdHFkbSh0b3RhbD10b3RhbCwgaW5pdGlhbD1pbml0aWFsLCBkZXNjPSJCQyBzZWF0cyIsIHVuaXQ9InNlYXQiKQogICAgaWRlbnRpdHkgPSBnZXRhdHRyKHJ1biwgImlkIiwgIm5vLXdhbmRiIikKICAgIHRhaWxlciA9IEpTT05MVGFpbGVyKG1ldHJpY3NfcGF0aCwgY3Vyc29yX3BhdGgsIGlkZW50aXR5PWlkZW50aXR5KQogICAgZGlzcGxheV90YWlsZXIgPSBKU09OTFRhaWxlcihtZXRyaWNzX3BhdGgsIGlkZW50aXR5PSJub3RlYm9vay1kaXNwbGF5IikKICAgIGxhdGVzdCA9IHt9CiAgICB2YWxpZGF0aW9ucyA9IFtdCiAgICBjb21wbGV0ZWQgPSBpbml0aWFsCiAgICBsb2dnaW5nX2Vycm9yID0gTm9uZQogICAgbW9uaXRvcl9zeW5jX2Vycm9ycyA9IDAKCiAgICBkZWYgc3luY19tb25pdG9yX21ldGFkYXRhKCk6CiAgICAgICAgbm9ubG9jYWwgbW9uaXRvcl9zeW5jX2Vycm9ycwogICAgICAgIGlmIG9uX3BvbGwgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIG9uX3BvbGwoKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGVycm9yOgogICAgICAgICAgICAgICAgbW9uaXRvcl9zeW5jX2Vycm9ycyArPSAxCiAgICAgICAgICAgICAgICBwcmludChmIuebkeaOp+WFg+aVsOaNruWQjOatpeWksei0pe+8iHt0eXBlKGVycm9yKS5fX25hbWVfX33vvInvvJvorq3nu4Pnu6fnu63vvIzmnKzmrKHlkIzmraXmnKrmiJDlip/jgIIiKQoKICAgIGNsYXNzIERlbGl2ZXJ5RmFpbHVyZShFeGNlcHRpb24pOgogICAgICAgIGRlZiBfX2luaXRfXyhzZWxmLCBvcmlnaW5hbCk6CiAgICAgICAgICAgIHNlbGYub3JpZ2luYWwgPSBvcmlnaW5hbAoKICAgIGRlZiBjb25zdW1lX2Rpc3BsYXkoKToKICAgICAgICBub25sb2NhbCBjb21wbGV0ZWQKICAgICAgICBwb3N0Zml4ID0gTm9uZQogICAgICAgIGZvciByZWNvcmQsIG9mZnNldCBpbiBkaXNwbGF5X3RhaWxlci5wZW5kaW5nKCk6CiAgICAgICAgICAgIHBoYXNlID0gcmVjb3JkLmdldCgicGhhc2UiKQogICAgICAgICAgICBpZiBwaGFzZSBpbiAoInN0YXRlX2JjX3RyYWluIiwgInN0YXRlX2JjX3ZhbGlkYXRpb24iKToKICAgICAgICAgICAgICAgIGxhdGVzdFtwaGFzZV0gPSByZWNvcmQKICAgICAgICAgICAgICAgIGlmIHBoYXNlID09ICJzdGF0ZV9iY192YWxpZGF0aW9uIjoKICAgICAgICAgICAgICAgICAgICB2YWxpZGF0aW9ucy5hcHBlbmQocmVjb3JkKQogICAgICAgICAgICAgICAgY291bnQgPSBjb21wbGV0ZWRfc2VhdHMocmVjb3JkLCBzZWF0c19wZXJfZXBvY2gpCiAgICAgICAgICAgICAgICBpZiBjb3VudCBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgICAgICBpZiBjb3VudCA+IHRvdGFsOgogICAgICAgICAgICAgICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUcmFpbmluZyBsb2cgY29udGFpbnMgZXBvY2hzIGJleW9uZCB0aGUgbm90ZWJvb2sgdGFyZ2V0IikKICAgICAgICAgICAgICAgICAgICBjb21wbGV0ZWQgPSBtYXgoY29tcGxldGVkLCBjb3VudCkKICAgICAgICAgICAgICAgICAgICBpZiBjb21wbGV0ZWQgPiBiYXIubjoKICAgICAgICAgICAgICAgICAgICAgICAgYmFyLnVwZGF0ZShjb21wbGV0ZWQgLSBiYXIubikKICAgICAgICAgICAgICAgIHBvc3RmaXggPSB7InBoYXNlIjogInZhbGlkYXRpb24iIGlmIHBoYXNlLmVuZHN3aXRoKCJ2YWxpZGF0aW9uIikgZWxzZSAidHJhaW5pbmciLAogICAgICAgICAgICAgICAgICAgICAgICAgICAiZXBvY2giOiByZWNvcmQuZ2V0KCJlcG9jaCIpLCAic3RlcCI6IHJlY29yZC5nZXQoIm9wdGltaXplcl9zdGVwcyIpfQogICAgICAgICAgICAgICAgaWYgaXNpbnN0YW5jZShyZWNvcmQuZ2V0KCJsb3NzIiksIChmbG9hdCwgaW50KSk6CiAgICAgICAgICAgICAgICAgICAgcG9zdGZpeFsibG9zcyJdID0gZiJ7cmVjb3JkWydsb3NzJ106LjRmfSIKICAgICAgICAgICAgICAgIGlmICJhbXBfc2NhbGUiIGluIHJlY29yZDoKICAgICAgICAgICAgICAgICAgICBwb3N0Zml4WyJzY2FsZSJdID0gcmVjb3JkWyJhbXBfc2NhbGUiXQogICAgICAgICAgICBkaXNwbGF5X3RhaWxlci5hY2tub3dsZWRnZShvZmZzZXQpCiAgICAgICAgaWYgcG9zdGZpeCBpcyBub3QgTm9uZToKICAgICAgICAgICAgYmFyLnNldF9wb3N0Zml4KHBvc3RmaXgpCgogICAgZGVmIGNvbnN1bWUoKToKICAgICAgICBmb3IgcmVjb3JkLCBvZmZzZXQgaW4gdGFpbGVyLnBlbmRpbmcoKToKICAgICAgICAgICAgcGF5bG9hZCA9IG1ldHJpY19wYXlsb2FkKHJlY29yZCkKICAgICAgICAgICAgaWYgcnVuIGlzIG5vdCBOb25lIGFuZCBwYXlsb2FkOgogICAgICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgICAgIHJ1bi5sb2cocGF5bG9hZCkgICMgRG8gbm90IHN1cHBseSBzdGVwPW9wdGltaXplcl9zdGVwcy4KICAgICAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXJyb3I6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgRGVsaXZlcnlGYWlsdXJlKGVycm9yKSBmcm9tIGVycm9yCiAgICAgICAgICAgIHRhaWxlci5hY2tub3dsZWRnZShvZmZzZXQpCgogICAgdHJ5OgogICAgICAgIHdoaWxlIFRydWU6CiAgICAgICAgICAgIGNvbnN1bWVfZGlzcGxheSgpICAjIEluZGVwZW5kZW50IG9mIFcmQiBjb25uZWN0aXZpdHkvY3Vyc29yIHN0YXRlLgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICBjb25zdW1lKCkKICAgICAgICAgICAgICAgIGxvZ2dpbmdfZXJyb3IgPSBOb25lCiAgICAgICAgICAgIGV4Y2VwdCBEZWxpdmVyeUZhaWx1cmUgYXMgZXJyb3I6CiAgICAgICAgICAgICAgICBpZiBsb2dnaW5nX2Vycm9yIGlzIE5vbmU6CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoIlcmQuaXpeW/l+aaguacqumAgei+vu+8jOiuree7g+e7p+e7re+8jOW+heWPkemAgeiusOW9leWwhuiHquWKqOmHjeivle+8m+ivt+WLv+mHjeWQr3RyYWluZXLjgIIiKQogICAgICAgICAgICAgICAgbG9nZ2luZ19lcnJvciA9IHR5cGUoZXJyb3Iub3JpZ2luYWwpLl9fbmFtZV9fCiAgICAgICAgICAgIHN5bmNfbW9uaXRvcl9tZXRhZGF0YSgpCiAgICAgICAgICAgIGNvZGUgPSBwcm9jZXNzLnBvbGwoKQogICAgICAgICAgICBpZiBjb2RlIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgY29uc3VtZV9kaXNwbGF5KCkKICAgICAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgICAgICBjb25zdW1lKCkgICMgRHJhaW4gZmluYWwgcmVjb3JkcyBhZnRlciB0aGUgY2hpbGQgaGFzIGZsdXNoZWQvZXhpdGVkLgogICAgICAgICAgICAgICAgICAgIGxvZ2dpbmdfZXJyb3IgPSBOb25lCiAgICAgICAgICAgICAgICBleGNlcHQgRGVsaXZlcnlGYWlsdXJlIGFzIGVycm9yOgogICAgICAgICAgICAgICAgICAgIGxvZ2dpbmdfZXJyb3IgPSB0eXBlKGVycm9yLm9yaWdpbmFsKS5fX25hbWVfXwogICAgICAgICAgICAgICAgc3luY19tb25pdG9yX21ldGFkYXRhKCkKICAgICAgICAgICAgICAgIGJyZWFrCiAgICAgICAgICAgIHRpbWUuc2xlZXAocG9sbF9zZWNvbmRzKQogICAgZXhjZXB0IEtleWJvYXJkSW50ZXJydXB0OgogICAgICAgIHByaW50KCLnm5Hmjqflt7LmmoLlgZzvvJt0cmFpbmVy5pyq6KKr5pys55uR5o6n5YGc5q2i44CC6YeN5paw6L+Q6KGM55uR5o6nY2VsbOWPr+e7p+e7reafpeeci+WQjOS4gOi/m+eoi+OAgiIpCiAgICAgICAgcmFpc2UKICAgIGZpbmFsbHk6CiAgICAgICAgYmFyLmNsb3NlKCkKICAgIGlmIGNvZGU6CiAgICAgICAgZmFpbHVyZV90YWlsID0gbG9nX3RhaWwodHJhaW5fbG9nX3BhdGgpCiAgICAgICAgcHJpbnQoZiJ0cmFpbmVyIHJldHVybmNvZGU6IHtjb2RlfVxuLS0tIHRyYWluaW5nLmxvZyB0YWlsIC0tLVxue2ZhaWx1cmVfdGFpbH0iKQogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcihmIkJDIHRyYWluZXIgZXhpdGVkIHdpdGggY29kZSB7Y29kZX0uXG5cbntmYWlsdXJlX3RhaWx9IikKICAgIGlmIGxvZ2dpbmdfZXJyb3I6CiAgICAgICAgcHJpbnQoInRyYWluZXLlt7LlrozmiJDvvIzkvYZXJkLku43mnInlvoXlj5HpgIHml6Xlv5fvvJtKU09OTOWPimN1cnNvcuW3suS/neeVme+8jOWPr+mHjeaWsOi/kOihjOebkeaOp+ihpeS8oOOAgiIpCiAgICByZXR1cm4geyJyZXR1cm5jb2RlIjogY29kZSwgImNvbXBsZXRlZF9zZWF0cyI6IGNvbXBsZXRlZCwgInRhcmdldF9zZWF0cyI6IHRvdGFsLAogICAgICAgICAgICAibGF0ZXN0IjogbGF0ZXN0LCAidmFsaWRhdGlvbl9ldmVudHMiOiBsZW4odmFsaWRhdGlvbnMpLCAibG9nX3Jlc2V0cyI6IHRhaWxlci5yZXNldHMsCiAgICAgICAgICAgICJsb2dnaW5nX3N0YXR1cyI6ICJwZW5kaW5nIiBpZiBsb2dnaW5nX2Vycm9yIGVsc2UgImFjY2VwdGVkIiwKICAgICAgICAgICAgImxvZ2dpbmdfZXJyb3JfdHlwZSI6IGxvZ2dpbmdfZXJyb3IsCiAgICAgICAgICAgICJtb25pdG9yX3N5bmNfZXJyb3JzIjogbW9uaXRvcl9zeW5jX2Vycm9yc30K","sha256":"0a980871452bcadebd9301c050aec16c6407daf746a6c033299dad7dacb063a1"},"notebook_state_bc_setup.py":{"base64":"...","sha256":"1254aeb4f8efc6a319b4810ad78a6727ebbbe938876a1b0fbd0d8169ea51abfa"},"notebook_state_bc_launch.py":{"base64":"IiIiT25lIGR1cmFibGUgQkMgdHJhaW5lciBwZXIgbm90ZWJvb2sgd29yayBkaXJlY3Rvcnk7IG5vIG1vZGVsIGltcG9ydHMuIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKaW1wb3J0IHRyYWNlYmFjawoKCmRlZiBhdG9taWNfanNvbihwYXRoLCB2YWx1ZSk6CiAgICBwYXRoID0gUGF0aChwYXRoKQogICAgcGF0aC5wYXJlbnQubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgdGVtcG9yYXJ5ID0gcGF0aC53aXRoX25hbWUocGF0aC5uYW1lICsgJy50bXAnKQogICAgdGVtcG9yYXJ5LndyaXRlX3RleHQoanNvbi5kdW1wcyh2YWx1ZSwgaW5kZW50PTIpLCBlbmNvZGluZz0ndXRmLTgnKQogICAgdGVtcG9yYXJ5LnJlcGxhY2UocGF0aCkKCgpkZWYgcHJvY2Vzc19pZGVudGl0eShwaWQpOgogICAgIiIiTGludXggUElEIHBsdXMgc3RhcnQgdGlja3MgcHJldmVudHMgYXR0YWNoaW5nIHRvIGEgcmV1c2VkIFBJRC4iIiIKICAgIHRyeToKICAgICAgICBmaWVsZHMgPSBQYXRoKGYnL3Byb2Mve2ludChwaWQpfS9zdGF0JykucmVhZF90ZXh0KCkucnNwbGl0KCcpJywgMSlbMV0uc3BsaXQoKQogICAgZXhjZXB0IChGaWxlTm90Rm91bmRFcnJvciwgUHJvY2Vzc0xvb2t1cEVycm9yLCBUeXBlRXJyb3IsIFZhbHVlRXJyb3IpOgogICAgICAgIHJldHVybiBOb25lCiAgICBpZiBmaWVsZHNbMF0gaW4gKCdaJywgJ1gnKToKICAgICAgICByZXR1cm4gTm9uZQogICAgcmV0dXJuIGZpZWxkc1sxOV0KCgpkZWYgYWxpdmUocmVjb3JkLCByb2xlKToKICAgIHBpZCwgc3RhcnQgPSByZWNvcmQuZ2V0KHJvbGUgKyAnX3BpZCcpLCByZWNvcmQuZ2V0KHJvbGUgKyAnX3N0YXJ0JykKICAgIHJldHVybiBib29sKHBpZCBhbmQgc3RhcnQgYW5kIHByb2Nlc3NfaWRlbnRpdHkocGlkKSA9PSBzdGFydCkKCgpkZWYgcmVhZF9zdGF0ZShwYXRoKToKICAgIHRyeToKICAgICAgICByZXR1cm4ganNvbi5sb2FkcyhQYXRoKHBhdGgpLnJlYWRfdGV4dChlbmNvZGluZz0ndXRmLTgnKSkKICAgIGV4Y2VwdCBGaWxlTm90Rm91bmRFcnJvcjoKICAgICAgICByZXR1cm4ge30KCgpjbGFzcyBUcmFpbmluZ0pvYjoKICAgICIiIlBvcGVuLWNvbXBhdGlibGUgcG9sbCBmb3IgYm90aCBvcmlnaW5hbCBhbmQgcmVhdHRhY2hlZCBub3RlYm9vayBrZXJuZWxzLiIiIgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHN0YXRlX3BhdGgpOgogICAgICAgIHNlbGYuc3RhdGVfcGF0aCA9IFBhdGgoc3RhdGVfcGF0aCkKCiAgICBAcHJvcGVydHkKICAgIGRlZiBwaWQoc2VsZik6CiAgICAgICAgcmVjb3JkID0gcmVhZF9zdGF0ZShzZWxmLnN0YXRlX3BhdGgpCiAgICAgICAgcmV0dXJuIHJlY29yZC5nZXQoJ3RyYWluZXJfcGlkJykgb3IgcmVjb3JkLmdldCgnd29ya2VyX3BpZCcpCgogICAgQHByb3BlcnR5CiAgICBkZWYgcmV0dXJuY29kZShzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5wb2xsKCkKCiAgICBkZWYgcG9sbChzZWxmKToKICAgICAgICByZWNvcmQgPSByZWFkX3N0YXRlKHNlbGYuc3RhdGVfcGF0aCkKICAgICAgICBpZiBhbGl2ZShyZWNvcmQsICd3b3JrZXInKSBvciBhbGl2ZShyZWNvcmQsICd0cmFpbmVyJyk6CiAgICAgICAgICAgIHJldHVybiBOb25lCiAgICAgICAgIyBOZXZlciBtaXN0YWtlIGEgZGlzYXBwZWFyZWQgc3VwZXJ2aXNvciAvIHJlY2xhaW1lZCBydW50aW1lIGZvciBzdWNjZXNzLgogICAgICAgIGlmIHJlY29yZC5nZXQoJ3N0YXRlJykgPT0gJ2V4aXRlZCcgYW5kIGlzaW5zdGFuY2UocmVjb3JkLmdldCgncmV0dXJuY29kZScpLCBpbnQpOgogICAgICAgICAgICByZXR1cm4gcmVjb3JkWydyZXR1cm5jb2RlJ10KICAgICAgICByZXR1cm4gMTI1CgoKZGVmIGxhdW5jaF9vcl9hdHRhY2goYXJndiwgKiwgc291cmNlX3Jvb3QsIHdvcmtzcGFjZSwgbG9nX3BhdGgsIGVudl9vdmVycmlkZXM9Tm9uZSk6CiAgICAiIiJSZXBlYXRlZCBsaXZlLWNlbGwgZXhlY3V0aW9uIGF0dGFjaGVzOyBhIGZhaWxlZCBqb2IgcmV0cmllcyBvbmx5IG9uIHJlcXVlc3QuIiIiCiAgICBpZiBzeXMucGxhdGZvcm0gIT0gJ2xpbnV4JzoKICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1RoaXMgQ29sYWIgcHJvY2VzcyBsYXVuY2hlciByZXF1aXJlcyBMaW51eCcpCiAgICBpbXBvcnQgZmNudGwKCiAgICBzb3VyY2Vfcm9vdCwgd29ya3NwYWNlID0gUGF0aChzb3VyY2Vfcm9vdCkucmVzb2x2ZSgpLCBQYXRoKHdvcmtzcGFjZSkucmVzb2x2ZSgpCiAgICB3b3Jrc3BhY2UubWtkaXIocGFyZW50cz1UcnVlLCBleGlzdF9vaz1UcnVlKQogICAgY29udHJvbCA9IHdvcmtzcGFjZSAvICcubm90ZWJvb2snCiAgICBjb250cm9sLm1rZGlyKGV4aXN0X29rPVRydWUpCiAgICBzdGF0ZV9wYXRoID0gY29udHJvbCAvICd0cmFpbmVyX3Byb2Nlc3MuanNvbicKICAgIGFyZ3YgPSBbc3RyKGFyZ3VtZW50KSBmb3IgYXJndW1lbnQgaW4gYXJndl0KICAgIGRpZ2VzdCA9IGhhc2hsaWIuc2hhMjU2KGpzb24uZHVtcHMoYXJndiwgc2VwYXJhdG9ycz0oJywnLCAnOicpKS5lbmNvZGUoKSkuaGV4ZGlnZXN0KCkKICAgIHdpdGggKGNvbnRyb2wgLyAnbGF1bmNoLmxvY2snKS5vcGVuKCdhJykgYXMgbG9jazoKICAgICAgICBmY250bC5mbG9jayhsb2NrLCBmY250bC5MT0NLX0VYKQogICAgICAgIHJlY29yZCA9IHJlYWRfc3RhdGUoc3RhdGVfcGF0aCkKICAgICAgICBhY3RpdmUgPSBhbGl2ZShyZWNvcmQsICd3b3JrZXInKSBvciBhbGl2ZShyZWNvcmQsICd0cmFpbmVyJykKICAgICAgICBpZiBhY3RpdmU6CiAgICAgICAgICAgIGlmIHJlY29yZC5nZXQoJ2FyZ3Zfc2hhMjU2JykgIT0gZGlnZXN0OgogICAgICAgICAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdBIHRyYWluZXIgaXMgYWxyZWFkeSBydW5uaW5nIHdpdGggZGlmZmVyZW50IGFyZ3VtZW50czsgYXR0YWNoIGZpcnN0JykKICAgICAgICAgICAgcHJpbnQoZidBdHRhY2hlZCB0byBleGlzdGluZyB0cmFpbmVyIFBJRCB7cmVjb3JkLmdldCgidHJhaW5lcl9waWQiKX0nLCBmbHVzaD1UcnVlKQogICAgICAgICAgICByZXR1cm4gVHJhaW5pbmdKb2Ioc3RhdGVfcGF0aCkKICAgICAgICBpZiByZWNvcmQuZ2V0KCdzdGF0ZScpID09ICdleGl0ZWQnIGFuZCByZWNvcmQuZ2V0KCdyZXR1cm5jb2RlJykgPT0gMCBhbmQgcmVjb3JkLmdldCgnYXJndl9zaGEyNTYnKSA9PSBkaWdlc3Q6CiAgICAgICAgICAgIHByaW50KCdUaGlzIHRyYWluaW5nIGNvbW1hbmQgYWxyZWFkeSBmaW5pc2hlZCBzdWNjZXNzZnVsbHknLCBmbHVzaD1UcnVlKQogICAgICAgICAgICByZXR1cm4gVHJhaW5pbmdKb2Ioc3RhdGVfcGF0aCkKICAgICAgICBzcGVjID0gewogICAgICAgICAgICAnYXJndic6IGFyZ3YsICdzb3VyY2Vfcm9vdCc6IHN0cihzb3VyY2Vfcm9vdCksICd3b3Jrc3BhY2UnOiBzdHIod29ya3NwYWNlKSwKICAgICAgICAgICAgJ2xvZ19wYXRoJzogc3RyKFBhdGgobG9nX3BhdGgpLnJlc29sdmUoKSksICdzdGF0ZV9wYXRoJzogc3RyKHN0YXRlX3BhdGgpLAogICAgICAgICAgICAnYXJndl9zaGEyNTYnOiBkaWdlc3QsCiAgICAgICAgICAgICdlbnZfb3ZlcnJpZGVzJzogZW52X292ZXJyaWRlcyBvciB7fSwKICAgICAgICB9CiAgICAgICAgc3BlY19wYXRoID0gY29udHJvbCAvICdsYXVuY2hfc3BlYy5qc29uJwogICAgICAgIGF0b21pY19qc29uKHNwZWNfcGF0aCwgc3BlYykKICAgICAgICB3aXRoIChjb250cm9sIC8gJ3N1cGVydmlzb3IubG9nJykub3BlbignYScsIGJ1ZmZlcmluZz0xKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgIHdvcmtlciA9IHN1YnByb2Nlc3MuUG9wZW4oCiAgICAgICAgICAgICAgICBbc3lzLmV4ZWN1dGFibGUsIHN0cihQYXRoKF9fZmlsZV9fKS5yZXNvbHZlKCkpLCAnLS13b3JrZXInLCBzdHIoc3BlY19wYXRoKV0sCiAgICAgICAgICAgICAgICBjd2Q9Jy9jb250ZW50JyBpZiBQYXRoKCcvY29udGVudCcpLmlzX2RpcigpIGVsc2Ugc3RyKHdvcmtzcGFjZSksCiAgICAgICAgICAgICAgICBzdGRvdXQ9b3V0cHV0LCBzdGRlcnI9c3VicHJvY2Vzcy5TVERPVVQsIHN0YXJ0X25ld19zZXNzaW9uPVRydWUsCiAgICAgICAgICAgICkKICAgICAgICAjIFRoZSB3b3JrZXIgcHVibGlzaGVzIGl0cyBvd24gaWRlbnRpdHkgYmVmb3JlIGltcG9ydGluZyAvIGxhdW5jaGluZyBUb3JjaC4KICAgICAgICBkZWFkbGluZSA9IHRpbWUubW9ub3RvbmljKCkgKyAxNQogICAgICAgIHdoaWxlIHRpbWUubW9ub3RvbmljKCkgPCBkZWFkbGluZToKICAgICAgICAgICAgcmVjb3JkID0gcmVhZF9zdGF0ZShzdGF0ZV9wYXRoKQogICAgICAgICAgICBpZiByZWNvcmQuZ2V0KCd3b3JrZXJfcGlkJykgPT0gd29ya2VyLnBpZDoKICAgICAgICAgICAgICAgIHByaW50KGYnVHJhaW5lciBzdXBlcnZpc29yIFBJRCB7d29ya2VyLnBpZH0nLCBmbHVzaD1UcnVlKQogICAgICAgICAgICAgICAgcmV0dXJuIFRyYWluaW5nSm9iKHN0YXRlX3BhdGgpCiAgICAgICAgICAgIGlmIHdvcmtlci5wb2xsKCkgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICAgICByYWlzZSBSdW50aW1lRXJyb3IoJ1RyYWluZXIgc3VwZXJ2aXNvciBmYWlsZWQ7IGluc3BlY3QgJyArIHN0cihjb250cm9sIC8gJ3N1cGVydmlzb3IubG9nJykpCiAgICAgICAgICAgIHRpbWUuc2xlZXAoLjEpCiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCdUcmFpbmVyIGlzIHN0aWxsIHN0YXJ0aW5nOyByZXJ1biB0aGlzIGNlbGwgdG8gYXR0YWNoLCBkbyBub3Qgc3RhcnQgYW5vdGhlciBjb21tYW5kJykKCgpkZWYgd29ya2VyX21haW4oc3BlY19wYXRoKToKICAgIGltcG9ydCBmY250bAoKICAgIHNwZWMgPSBqc29uLmxvYWRzKFBhdGgoc3BlY19wYXRoKS5yZWFkX3RleHQoZW5jb2Rpbmc9J3V0Zi04JykpCiAgICB3b3Jrc3BhY2UgPSBQYXRoKHNwZWNbJ3dvcmtzcGFjZSddKQogICAgc3RhdGVfcGF0aCA9IFBhdGgoc3BlY1snc3RhdGVfcGF0aCddKQogICAgcmVjb3JkID0gewogICAgICAgICdhcmd2X3NoYTI1Nic6IHNwZWNbJ2FyZ3Zfc2hhMjU2J10sICd3b3JrZXJfcGlkJzogb3MuZ2V0cGlkKCksCiAgICAgICAgJ3dvcmtlcl9zdGFydCc6IHByb2Nlc3NfaWRlbnRpdHkob3MuZ2V0cGlkKCkpLCAnc3RhdGUnOiAnc3RhcnRpbmcnLAogICAgICAgICdzdGFydGVkX3VuaXgnOiB0aW1lLnRpbWUoKSwgJ3JldHVybmNvZGUnOiBOb25lLAogICAgfQogICAgbG9ja19wYXRoID0gd29ya3NwYWNlIC8gJy5ub3RlYm9vay90cmFpbmVyLmxvY2snCiAgICB3aXRoIGxvY2tfcGF0aC5vcGVuKCdhJykgYXMgbG9jazoKICAgICAgICBmY250bC5mbG9jayhsb2NrLCBmY250bC5MT0NLX0VYIHwgZmNudGwuTE9DS19OQikKICAgICAgICBhdG9taWNfanNvbihzdGF0ZV9wYXRoLCByZWNvcmQpCiAgICAgICAgY2hpbGQgPSBOb25lCiAgICAgICAgdHJ5OgogICAgICAgICAgICBsb2dfcGF0aCA9IFBhdGgoc3BlY1snbG9nX3BhdGgnXSkKICAgICAgICAgICAgbG9nX3BhdGgucGFyZW50Lm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKICAgICAgICAgICAgZW52aXJvbm1lbnQgPSB7Kipvcy5lbnZpcm9uLCAqKnNwZWNbJ2Vudl9vdmVycmlkZXMnXSwgJ1BZVEhPTlVOQlVGRkVSRUQnOiAnMSd9CiAgICAgICAgICAgIHdpdGggbG9nX3BhdGgub3BlbignYScsIGJ1ZmZlcmluZz0xKSBhcyBvdXRwdXQ6CiAgICAgICAgICAgICAgICBjaGlsZCA9IHN1YnByb2Nlc3MuUG9wZW4oc3BlY1snYXJndiddLCBjd2Q9c3BlY1snc291cmNlX3Jvb3QnXSwgZW52PWVudmlyb25tZW50LAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHN0ZG91dD1vdXRwdXQsIHN0ZGVycj1zdWJwcm9jZXNzLlNURE9VVCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBzdGFydF9uZXdfc2Vzc2lvbj1UcnVlKQogICAgICAgICAgICAgICAgcmVjb3JkLnVwZGF0ZShzdGF0ZT0ncnVubmluZycsIHRyYWluZXJfcGlkPWNoaWxkLnBpZCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdHJhaW5lcl9zdGFydD1wcm9jZXNzX2lkZW50aXR5KGNoaWxkLnBpZCkpCiAgICAgICAgICAgICAgICBhdG9taWNfanNvbihzdGF0ZV9wYXRoLCByZWNvcmQpCiAgICAgICAgICAgICAgICByZXR1cm5jb2RlID0gY2hpbGQud2FpdCgpCiAgICAgICAgZXhjZXB0IEJhc2VFeGNlcHRpb246CiAgICAgICAgICAgICMgQSBsaXZlIGNoaWxkIHJlbWFpbnMgZGlzY292ZXJhYmxlOyBkbyBub3QgZmFsc2VseSByZWNvcmQgc3VjY2Vzcy4KICAgICAgICAgICAgdHJhY2ViYWNrLnByaW50X2V4YygpCiAgICAgICAgICAgIHJldHVybmNvZGUgPSAxMjUKICAgICAgICByZWNvcmQudXBkYXRlKHN0YXRlPSdleGl0ZWQnLCByZXR1cm5jb2RlPXJldHVybmNvZGUsIGZpbmlzaGVkX3VuaXg9dGltZS50aW1lKCkpCiAgICAgICAgYXRvbWljX2pzb24oc3RhdGVfcGF0aCwgcmVjb3JkKQoKCmlmIF9fbmFtZV9fID09ICdfX21haW5fXyc6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcihkZXNjcmlwdGlvbj1fX2RvY19fKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgnLS13b3JrZXInLCB0eXBlPVBhdGgsIHJlcXVpcmVkPVRydWUpCiAgICB3b3JrZXJfbWFpbihwYXJzZXIucGFyc2VfYXJncygpLndvcmtlcikK","sha256":"3b5d4b7c90400faa54d6fb2679d5bc9e1c3e550cd2df8b404a7ad29a0db8139d"}}
for name, record in EMBEDDED_MODULES.items():
    raw = base64.b64decode(record['base64'])
    if hashlib.sha256(raw).hexdigest() != record['sha256']:
        raise RuntimeError('notebook工具校验失败: ' + name)
    (TOOLS / name).write_bytes(raw)
if str(TOOLS) not in sys.path:
    sys.path.insert(0, str(TOOLS))
from notebook_state_bc_setup import copy_verified, extract_source, stage_dataset, stage_resume, build_trainer_argv, preflight
from notebook_state_bc import monitor_process, resume_completed, configure_wandb, log_tail
from notebook_state_bc_launch import launch_or_attach, TrainingJob, atomic_json
total_ram = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1024**3
RAM_LIMIT_GIB = round(max(3, min(40, total_ram * .6)), 1)
print('CUDA:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
print('主机内存保护:', RAM_LIMIT_GIB, 'GiB')


## 代码3 / 6：W&B在线登录

推荐在Colab左侧Secrets保存`WANDB_API_KEY`并允许本notebook访问；也可在下方密码框输入。
不要把密钥写进单元格或聊天。登录/连接失败会明确报错，不会把离线日志冒充在线记录。
页面会打印本次W&B运行链接。重复监控和重连运行时会复用同一个run ID。


In [ ]:
import getpass, uuid
import wandb
from IPython.display import display, Markdown

identity_path = NOTEBOOK_RUN / 'wandb_run.json'
if identity_path.exists():
    wb_identity = json.loads(identity_path.read_text())
    if wb_identity['project'] != WANDB_PROJECT or wb_identity.get('requested_entity') != WANDB_ENTITY:
        raise ValueError('该续训目录已有不同的W&B project/entity；请沿用原设置或另选NOTEBOOK_RUN。')
else:
    wb_identity = {'id': uuid.uuid4().hex[:12], 'project': WANDB_PROJECT,
                   'requested_entity': WANDB_ENTITY, 'name': WANDB_NAME}
    atomic_json(identity_path, wb_identity)

def ensure_wandb():
    current = wandb.run
    if current is not None:
        if current.id != wb_identity['id']:
            raise RuntimeError('已有另一个W&B run；请先结束它再连接本次运行。')
        return current
    if not globals().get('_bc_wandb_logged_in', False):
        key = os.environ.get('WANDB_API_KEY')
        if not key:
            try:
                from google.colab import userdata
                key = userdata.get('WANDB_API_KEY')
            except Exception:
                key = None
        if not key:
            key = getpass.getpass('W&B API Key（输入隐藏）: ').strip()
        if not key:
            raise ValueError('需要W&B登录后再开训，尚未启动trainer。')
        try:
            authenticated = wandb.login(key=key, relogin=True)
        finally:
            del key
        if not authenticated:
            raise RuntimeError('W&B登录未成功，尚未启动trainer。')
    run = wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, id=wb_identity['id'],
                     resume='allow', name=wb_identity['name'], mode='online',
                     dir=str(WORK), save_code=False, settings=wandb.Settings(init_timeout=90),
                     config={'decision_seconds': 0.4, 'source_run': str(SOURCE_RUN),
                             'target_total_epochs': TARGET_EPOCHS})
    configure_wandb(run)
    globals()['_bc_wandb_logged_in'] = True
    return run

wb_run = ensure_wandb()
display(Markdown('W&B：[' + wb_run.url + '](' + wb_run.url + ')'))


## 代码4 / 6：最小缓存、自动恢复、启动前检查

只复制数据manifest引用的文件，跳过audit、历史模型、原始重建资料。
当前训练器校验完整manifest，因此所有引用shard都要保留；不会下载整棵Drive目录。
已校验的本地缓存直接复用。候选checkpoint只预览小型元数据，选出最新有效版本后才复制一个完整文件。
模型、Adam、loss设置和数据指纹保持原阶段；来源不匹配会停止，不自动从零初始化。


In [ ]:
from tqdm.auto import tqdm
marker = WORK / '.notebook/trainer_process.json'
if marker.exists() and TrainingJob(marker).poll() is None:
    raise RuntimeError('trainer仍在运行；请直接重跑代码5查看进度，避免重做准备。')
archive_candidates = [NOTEBOOK_RUN / 'source.tar.gz', SOURCE_RUN / 'source_recovery_revision2.tar.gz',
                      SOURCE_RUN / 'source.tar.gz']
archive_drive = next((path for path in archive_candidates if path.is_file()), None)
if archive_drive is None:
    raise FileNotFoundError('Drive中未找到本阶段源码包。')
archive_local = WORK / 'source.tar.gz'
copy_verified(archive_drive, archive_local)
SRC = extract_source(archive_local, WORK)
manifest_drive = json.loads((SOURCE_RUN / 'dataset/dataset.manifest.json').read_text())
with tqdm(total=len(manifest_drive['shards']), desc='缓存BC数据', unit='shard') as data_bar:
    def data_progress(done, total, relative):
        data_bar.update(done - data_bar.n)
    DATASET = stage_dataset(SOURCE_RUN, WORK, progress=data_progress)
del manifest_drive
print('选择最新完整checkpoint…')
staged = stage_resume(SOURCE_RUN, NOTEBOOK_RUN, WORK, manifest=DATASET,
                      extra_runs=[SOURCE_RUN / 'notebook_bc'])
CHECKPOINT = staged['checkpoint']
TRAIN = staged['out']
info = staged['info']
SEATS_PER_EPOCH = info['train_seats_per_epoch']
INITIAL_COMPLETED = resume_completed(info['progress'], SEATS_PER_EPOCH)
if INITIAL_COMPLETED > TARGET_EPOCHS * SEATS_PER_EPOCH:
    raise ValueError('恢复点已超过目标epoch；请在代码1提高TARGET_EPOCHS，然后重跑代码4。')
registry = SRC / 'fixtures/split_registry.json'
if not registry.is_file():
    raise FileNotFoundError('缺少原对局划分registry，拒绝改变训练/验证分区。')
COMMAND = build_trainer_argv(info, source_root=SRC, dataset_manifest=DATASET,
                            checkpoint=CHECKPOINT, out=TRAIN,
                            backup_dir=NOTEBOOK_RUN / 'training', epochs=TARGET_EPOCHS,
                            ram_limit_gib=RAM_LIMIT_GIB, split_registry=registry,
                            lr_schedule='cosine', warmup_steps=WARMUP_STEPS,
                            min_learning_rate=MIN_LEARNING_RATE,
                            frame_microbatch=FRAME_MICROBATCH, defer_metrics=DEFER_METRICS,
                            fused_optimizer=FUSED_OPTIMIZER,
                            float32_matmul_precision=FLOAT32_MATMUL_PRECISION,
                            checkpoint_every_seats=CHECKPOINT_EVERY_SEATS,
                            validation_every_seats=VALIDATION_EVERY_SEATS)
TRAIN_ENV = {'PYTHONPATH': str(SRC / 'src'), 'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2',
             'PYTHONDONTWRITEBYTECODE': '1'}
print('恢复来源:', staged['source_run'])
print('恢复进度:', info['progress'], '| 本阶段完成座位:', INITIAL_COMPLETED,
      '/', TARGET_EPOCHS * SEATS_PER_EPOCH)
if staged['rejected_candidates']:
    print('已跳过不完整/不匹配候选:', staged['rejected_candidates'])
print('启动检查:', preflight(COMMAND, SRC, require_cuda=True, env=TRAIN_ENV))
wb_run = ensure_wandb()
wb_run.config.update({'dataset_fingerprint': info['dataset_fingerprint'],
                      'target_total_epochs': TARGET_EPOCHS,
                      'resume_optimizer_step': info['progress']['optimizer_steps'],
                      'resume_epoch': info['progress']['epoch'],
                      'train_seats_per_epoch': SEATS_PER_EPOCH,
                      'source_archive_sha256': hashlib.sha256(archive_local.read_bytes()).hexdigest()},
                     allow_val_change=True)
print('准备完成；运行代码5开始或连接训练。')


## 代码5 / 6：训练 + W&B + tqdm（可重复运行）

打开W&B链接看曲线；下方进度条按完整轨迹座位计数，显示全局step和loss。
停止本单元格只暂停显示，trainer仍在后台；重跑本单元格可重新连接。
训练异常退出会打印真实`training.log`尾部。修正启动问题后，从代码4重新检查、恢复再运行。


In [ ]:
wb_run = ensure_wandb()
cursor = WORK / '.notebook/wandb_cursor.json'
remote_cursor = NOTEBOOK_RUN / 'wandb_cursor.json'
if not cursor.exists() and remote_cursor.is_file():
    copy_verified(remote_cursor, cursor)
sync_clock = {'last': 0.0}
def sync_small_files(force=False):
    if not force and time.monotonic() - sync_clock['last'] < 120:
        return
    if cursor.is_file():
        copy_verified(cursor, remote_cursor)
    sync_clock['last'] = time.monotonic()
job = launch_or_attach(COMMAND, source_root=SRC, workspace=WORK,
                       log_path=TRAIN / 'training.log', env_overrides=TRAIN_ENV)
print('Trainer PID:', job.pid, '| W&B:', wb_run.url)
try:
    result = monitor_process(job, TRAIN / 'metrics.jsonl', TRAIN / 'training.log',
                             run=wb_run, target_epochs=TARGET_EPOCHS,
                             seats_per_epoch=SEATS_PER_EPOCH,
                             initial_completed=INITIAL_COMPLETED,
                             cursor_path=cursor, poll_seconds=2,
                             on_poll=sync_small_files)
    try:
        sync_small_files(force=True)
    except Exception as error:
        print('小型日志游标尚未同步:', type(error).__name__, '；训练checkpoint由trainer独立备份。')
    print(result)
    if result['logging_status'] == 'accepted':
        wb_run.finish(exit_code=0)
    else:
        print('W&B仍有待发送记录，保持run打开；稍后重跑代码5补传。')
except KeyboardInterrupt:
    print('需要继续显示时重跑代码5。')
    raise
except Exception:
    if job.poll() is not None:
        wb_run.finish(exit_code=1)
    raise


## 代码6 / 6：查看最近验证与进程状态

可在暂停监控后运行。展示轨迹标签验证；实战胜率仍需固定对手对战评估。
为减少首次加载，这版BC notebook不预载旧BC或旧快照评测模型，也不安装模拟器。


In [ ]:
import html
from IPython.display import HTML
metrics_file = TRAIN / 'metrics.jsonl'
validations = []
if metrics_file.is_file():
    for line in metrics_file.read_text(errors='replace').splitlines():
        try:
            row = json.loads(line)
        except json.JSONDecodeError:
            continue
        if row.get('phase') == 'state_bc_validation':
            validations.append(row)
columns = [('epoch', 'Epoch'), ('optimizer_steps', 'Step'), ('loss', 'Loss'),
           ('slot_accuracy', '选牌准确率'), ('act_recall', '动作召回')]
table = '<table><tr>' + ''.join('<th>' + label + '</th>' for _, label in columns) + '</tr>'
for row in validations[-10:]:
    table += '<tr>'
    for key, label in columns:
        value = row.get(key, '')
        if key in ('slot_accuracy', 'act_recall') and isinstance(value, (int, float)):
            value = f'{value:.2%}'
        elif key == 'loss' and isinstance(value, (int, float)):
            value = f'{value:.4f}'
        table += '<td>' + html.escape(str(value)) + '</td>'
    table += '</tr>'
display(HTML(table + '</table>'))
if marker.exists():
    current_job = TrainingJob(marker)
    print('Trainer状态:', '运行中' if current_job.poll() is None else '已退出，returncode=' + str(current_job.poll()))
else:
    print('Trainer状态: 尚未启动，运行代码5开始训练。')
print('本地checkpoint:', TRAIN / 'latest.pt')
print('Drive checkpoint:', NOTEBOOK_RUN / 'training/latest.pt')
print('W&B:', wb_run.url)
if not validations:
    print('尚无完整验证结果；启动或首个验证阶段可能仍在进行。')
